In [1]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split

print("CUDA available :", torch.cuda.is_available())
print("GPU name      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch version:", torch.__version__)

CUDA available : True
GPU name      : Tesla T4
PyTorch version: 2.10.0+cu128


In [4]:
# Click the folder icon on the left → Upload → choose Sarcasm_Headlines_Dataset.json

# Then run this cell
df = pd.read_json('/content/Sarcasm_Headlines_Dataset.json', lines=True)

print("Shape:", df.shape)
print(df['is_sarcastic'].value_counts(normalize=True))
df.head(3)

Shape: (26709, 3)
is_sarcastic
0    0.561047
1    0.438953
Name: proportion, dtype: float64


,article_link,headline,is_sarcastic
0,https://www.huffingtonpost.com/entry/versace-b...,former versace store clerk sues over secret 'b...,0
1,https://www.huffingtonpost.com/entry/roseanne-...,the 'roseanne' revival catches up to our thorn...,0
2,https://local.theonion.com/mom-starting-to-fea...,mom starting to fear son's web series closest ...,1


In [5]:
df = df[["headline", "is_sarcastic"]]

In [6]:
df

,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0
...,...,...
26704,american politics in moral free-fall,0
26705,america's best 20 hikes,0
26706,reparations and obama,0
26707,israeli ban targeting boycott supporters raise...,0


In [8]:
train_df, temp = train_test_split(
    df,
    test_size=0.2,
    stratify=df['is_sarcastic'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp,
    test_size=0.5,
    stratify=temp['is_sarcastic'],
    random_state=42
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# Hugging Face Dataset format + rename column to 'label'
dataset = DatasetDict({
    'train':       Dataset.from_pandas(train_df[['headline', 'is_sarcastic']].rename(columns = {'is_sarcastic': 'label'})),
    'validation':  Dataset.from_pandas(val_df[['headline', 'is_sarcastic']].rename(columns = {'is_sarcastic': 'label'})),
    'test':        Dataset.from_pandas(test_df[['headline', 'is_sarcastic']].rename(columns = {'is_sarcastic': 'label'})),
})

Train: (21367, 2)
Val  : (2671, 2)
Test : (2671, 2)


In [9]:
model_name = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["headline"],
        truncation=True,
        max_length=96,
        padding="max_length"
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/21367 [00:00<?, ? examples/s]

Map:   0%|          | 0/2671 [00:00<?, ? examples/s]

Map:   0%|          | 0/2671 [00:00<?, ? examples/s]

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Move to GPU explicitly (good practice)
if torch.cuda.is_available():
    model = model.cuda()

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir                  = "/content/sarcasm_checkpoints",
    num_train_epochs            = 4,
    per_device_train_batch_size = 24,           # should fit T4
    per_device_eval_batch_size  = 48,
    gradient_accumulation_steps = 2,            # effective batch ~48
    learning_rate               = 3e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    fp16                        = True,         # mixed precision → faster & less memory
    eval_strategy               = "epoch",      # Uncommented to match save_strategy
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    greater_is_better           = True,
    logging_steps               = 200,
    report_to                   = "none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [18]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized_datasets["train"],
    eval_dataset    = tokenized_datasets["validation"],
    # tokenizer       = tokenizer, # Removed as it's not a recognized argument
    compute_metrics = compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.544981,0.245135,0.898165
2,0.345256,0.190890,0.918757
3,0.207180,0.248889,0.919506
4,0.148013,0.312502,0.922875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1784, training_loss=0.35295985204756525, metrics={'train_runtime': 381.9702, 'train_samples_per_second': 223.756, 'train_steps_per_second': 4.671, 'total_flos': 2122823180312064.0, 'train_loss': 0.35295985204756525, 'epoch': 4.0})

In [19]:
test_results = trainer.evaluate(tokenized_datasets["test"])
print("\nFinal test performance:")
print(test_results)


Final test performance:
{'eval_loss': 0.30703794956207275, 'eval_accuracy': 0.9239985024335455, 'eval_runtime': 1.8591, 'eval_samples_per_second': 1436.686, 'eval_steps_per_second': 30.121, 'epoch': 4.0}


In [20]:
SAVE_PATH = "/content/sarcasm_model"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved to: {SAVE_PATH}")
print("Now you can right-click the folder → Download → zip and use in Streamlit")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/sarcasm_model
Now you can right-click the folder → Download → zip and use in Streamlit


In [21]:
!zip -r sarcasm_model.zip /content/sarcasm_model
from google.colab import files
files.download('sarcasm_model.zip')

  adding: content/sarcasm_model/ (stored 0%)
  adding: content/sarcasm_model/config.json (deflated 51%)
  adding: content/sarcasm_model/training_args.bin (deflated 53%)
  adding: content/sarcasm_model/tokenizer.json (deflated 82%)
  adding: content/sarcasm_model/tokenizer_config.json (deflated 50%)
  adding: content/sarcasm_model/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>